# Exercițiu Laborator 2 — Formula lui Bayes

## Enunțul problemei
Investigăm o boală ($B$) care afectează $1\%$ din populație: $P(B) = 0.01$ (deci $P(\neg B) = 0.99$). Un test de diagnosticare are următoarele caracteristici:
* **Sensibilitate:** $P(\text{Test}=\text{Pozitiv} \mid B) = 0.95$;
* **Specificitate:** $P(\text{Test}=\text{Negativ} \mid \neg B) = 0.90$, ceea ce implică o rată de rezultate fals pozitive de:
  $$P(\text{Test}=\text{Pozitiv} \mid \neg B) = 1 - 0.90 = 0.10.$$

**Cerințe:**
* **a)** Dacă o persoană este testată și rezultatul este pozitiv, care este probabilitatea ca aceasta să fie efectiv bolnavă? Cum explicați rezultatul?
* **b)** Modelați în Python probabilitatea de mai sus și comparați cu cea obținută teoretic la punctul anterior.
* **c)** Care ar trebui să fie specificitatea minimă pentru ca probabilitatea de mai sus să ajungă la $50\%$?

## a) Calculul probabilității a posteriori și interpretarea rezultatului

Notăm evenimentul $T^+ = \{\text{Test} = \text{Pozitiv}\}$. Conform **Formulei lui Bayes**:
$$
P(B \mid T^+) = \frac{P(T^+ \mid B) \cdot P(B)}{P(T^+)}
$$
unde probabilitatea totală a unui test pozitiv este:
$$
P(T^+) = P(T^+ \mid B)P(B) + P(T^+ \mid \neg B)P(\neg B) = 0.95 \times 0.01 + 0.10 \times 0.99 = 0.0095 + 0.099 = 0.1085.
$$

Înlocuind în formula lui Bayes obținem:
$$
P(B \mid T^+) = \frac{0.0095}{0.1085} = \frac{95}{1085} = \frac{19}{217} \approx 0.087558 \quad (8.76\%).
$$

### Explicația rezultatului (Paradoxul fals-pozitivelor / *Base-rate fallacy*)
Deși testul are o acuratețe aparent ridicată ($95\%$ sensibilitate și $90\%$ specificitate), prevalența bolii în populație este foarte mică ($1\%$). Într-un eșantion reprezentativ de $10\,000$ de persoane:
* $100$ persoane sunt bolnave, dintre care $95$ primesc test pozitiv (**adevărat pozitive**);
* $9\,900$ persoane sunt sănătoase, dintre care $10\%$, adică $990$ de persoane, primesc eronat test pozitiv (**fals pozitive**).

Astfel, din totalul celor $95 + 990 = 1\,085$ persoane testate pozitiv, majoritatea covârșitoare ($990$) sunt sănătoase, iar doar $\frac{95}{1085} \approx 8.76\%$ sunt efectiv bolnave.

## b) Modelarea în Python și comparația cu rezultatul teoretic

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parametrii problemei
P_B = 0.01
P_not_B = 1 - P_B
sensibilitate = 0.95   # P(T+ | B)
specificitate = 0.90   # P(T- | ~B)
P_T_poz_given_not_B = 1 - specificitate

# 1. Calcul analitic exact în Python
P_T_poz = sensibilitate * P_B + P_T_poz_given_not_B * P_not_B
P_B_given_T_poz_teoretic = (sensibilitate * P_B) / P_T_poz

# 2. Simulare Monte Carlo pe o populație de N = 2.000.000 persoane
rng = np.random.default_rng(42)
N = 2_000_000

este_bolnav = rng.random(N) < P_B
test_pozitiv = np.where(
    este_bolnav,
    rng.random(N) < sensibilitate,
    rng.random(N) < P_T_poz_given_not_B
)

P_B_given_T_poz_simulat = np.mean(este_bolnav[test_pozitiv])

print(f"P(B | T+) teoretic (Bayes): {P_B_given_T_poz_teoretic:.6f} ({P_B_given_T_poz_teoretic*100:.2f}%)")
print(f"P(B | T+) simulat (MC):     {P_B_given_T_poz_simulat:.6f} ({P_B_given_T_poz_simulat*100:.2f}%)")
print(f"Diferența absolută:         {abs(P_B_given_T_poz_teoretic - P_B_given_T_poz_simulat):.6f}")

## c) Determinarea specificității minime pentru $P(B \mid T^+) \ge 50\%$

Fie $s = P(T^- \mid \neg B) \in [0, 1]$ specificitatea necunoscută, deci probabilitatea unui rezultat fals pozitiv este $P(T^+ \mid \neg B) = 1 - s$. Impunem condiția:
$$
P(B \mid T^+) = \frac{P(T^+ \mid B) P(B)}{P(T^+ \mid B) P(B) + (1 - s) P(\neg B)} \ge 0.50.
$$

Înlocuind valorile cunoscute:
$$
\frac{0.95 \times 0.01}{0.95 \times 0.01 + (1 - s) \times 0.99} \ge \frac{1}{2}
$$
$$
\frac{0.0095}{0.0095 + 0.99(1 - s)} \ge \frac{1}{2} \iff 2 \times 0.0095 \ge 0.0095 + 0.99(1 - s)
$$
$$
0.99(1 - s) \le 0.0095 \iff 1 - s \le \frac{0.0095}{0.99} = \frac{19}{1980} \approx 0.009596
$$
$$
s \ge 1 - \frac{19}{1980} = \frac{1961}{1980} \approx 0.990404 \quad (99.04\%).
$$

**Concluzie:** Specificitatea testului trebuie să fie de cel puțin **$99.04\%$** pentru ca un rezultat pozitiv să indice o probabilitate de cel puțin $50\%$ ca persoana să fie bolnavă.

In [ ]:
# Verificare numerică și vizualizare grafică a dependenței P(B | T+) de specificitate
s_min = 1 - (sensibilitate * P_B) / P_not_B
print(f"Specificitatea minimă calculată: s_min = {s_min:.6f} ({s_min*100:.4f}%)")

s_vals = np.linspace(0.85, 1.0, 500)
post_vals = (sensibilitate * P_B) / (sensibilitate * P_B + (1 - s_vals) * P_not_B)

plt.figure(figsize=(9, 4.5))
plt.plot(s_vals * 100, post_vals * 100, color="#1f77b4", linewidth=2, label="P(B | Test = Pozitiv)")
plt.axhline(50, color="#d62728", linestyle="--", label="Prag țintă: 50%")
plt.axvline(s_min * 100, color="#2ca02c", linestyle=":", linewidth=1.8, label=f"Specificitate minimă ≈ {s_min*100:.2f}%")
plt.scatter([s_min * 100], [50], color="#d62728", zorder=5)
plt.xlabel("Specificitate (%)")
plt.ylabel("P(Boală | Test Pozitiv) (%)")
plt.title("Probabilitatea a posteriori a bolii în funcție de specificitatea testului")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()